In [1]:
import sys
print(sys.executable)

import subprocess
import os

BASE_DIR = os.getcwd()
INPUT_DIRECTORY = os.path.join(BASE_DIR, "images", "original")
OUTPUT_BASE_DIRECTORY = os.path.join(BASE_DIR,  "images", "processed")
OUTPUT_MPI_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "mpi")
OUTPUT_OMP_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "omp")
OUTPUT_HYB_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "hyb")
OUTPUT_CUDA_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "cuda")

print(INPUT_DIRECTORY)
print(os.listdir(INPUT_DIRECTORY))

import re

from PIL import Image

/usr/local/Anaconda3-2025.06/bin/python
/users/eleves-a/2024/toan.lopez/parallel-gif-filter/images/original
['051009.vince.gif', '1.gif', '200_s.gif', '9815573.gif', 'Campusplan-Hausnr.gif', 'Campusplan-Mobilitaetsbeschraenkte.gif', 'Mandelbrot-large.gif', 'Produits_sous_linux.gif', 'TimelyHugeGnu.gif', 'australian-flag-large.gif', 'fire.gif', 'frame_002.gif', 'giphy-3.gif', 'porsche.gif', 'tumblr_myxlbtwVEb1qzt4vjo1_r14_500_large.gif', 'walla.gif', 'wallace.gif', 'walle.gif']


In [2]:
def run_command(cmd, env=None):
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env, shell=True)
    return result.stdout.decode()

# MPI
def run_sobelf_mpi(input_file, output_file, ranks=16, nodes=1, input_dir=INPUT_DIRECTORY):
    input_path = os.path.join(input_dir, input_file)
    output_path = os.path.join(OUTPUT_MPI_DIRECTORY, output_file)
    
    cmd = f"salloc -N {nodes} -n {ranks} mpirun ./sobelf_mpi {input_path} {output_path}"
    return run_command(cmd)

# OpenMP
def run_sobelf_omp(input_file, output_file, threads=16, input_dir=INPUT_DIRECTORY):
    input_path = os.path.join(input_dir, input_file)
    output_path = os.path.join(OUTPUT_OMP_DIRECTORY, output_file)

    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = str(threads)
    cmd = f"salloc -n 1 ./sobelf_omp {input_path} {output_path}"
    return run_command(cmd, env=env)

# OpenMP + MPI
def run_sobelf_hyb(input_file, output_file, ranks=4, threads=4, nodes=1, input_dir=INPUT_DIRECTORY):
    input_path = os.path.join(input_dir, input_file)
    output_path = os.path.join(OUTPUT_HYB_DIRECTORY, output_file)

    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = str(threads)
    cmd = f"salloc -N {nodes} -n {ranks} ./sobelf_hyb {input_path} {output_path}"
    return run_command(cmd, env=env)

# CUDA
def run_sobelf_cuda(input_file, output_file, input_dir=INPUT_DIRECTORY):
    input_path = os.path.join(input_dir, input_file)
    output_path = os.path.join(OUTPUT_CUDA_DIRECTORY, output_file)
    cmd = f"salloc -N 1 -n 1 ./sobelf_cuda {input_path} {output_path}"
    return run_command(cmd)



In [3]:

# Parse output
def parse_sobel_filter_output(output):
    match = re.search(r"SOBEL done in ([0-9.]+) s", output)
    if match:
        return float(match.group(1))
    else:
        raise ValueError("couldn't parse output :\n" + output + '\n')

def profile_mpi(input_file, output_file, ranks=16, nodes=1):
    return parse_sobel_filter_output(run_sobelf_mpi(input_file, output_file, ranks, nodes))

def profile_omp(input_file, output_file, threads=16):
    return parse_sobel_filter_output(run_sobelf_omp(input_file, output_file, threads))

def profile_hyb(input_file, output_file, ranks=4, threads=4, nodes=1):
    return parse_sobel_filter_output(run_sobelf_hyb(input_file, output_file, ranks, threads, nodes))

def profile_cuda(input_file, output_file):
    return parse_sobel_filter_output(run_sobelf_mpi(input_file, output_file))


In [4]:
def get_gif_info(path, input_dir=INPUT_DIRECTORY):
    sizes = []
    with Image.open(os.path.join(input_dir, path)) as im:
        num_frames = im.n_frames
        for frame_index in range(num_frames):
            im.seek(frame_index)
            sizes.append(im.size)
    return num_frames, sizes

In [5]:
print(profile_omp("1.gif", "1-sobel.gif"))
print(get_gif_info("1.gif"))

0.017806
(10, [(500, 500), (500, 500), (500, 500), (500, 500), (500, 500), (500, 500), (500, 500), (500, 500), (500, 500), (500, 500)])


In [6]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

from tqdm import tqdm

feature_names = ["num_frames","width","height","ranks","threads","nodes","cuda_available"]

def create_data_set(input_dir, rank_list, thread_list, node_list, cuda_list):
    # X : num_frames, width, height, ranks, threads, nodes, cuda_available
    # y : {0 : MPI , 1 : OMP , 2 : HYB , 3 : CUDA}
        
    X = []
    y = []

    for gif_file in tqdm(os.listdir(input_dir)):
        if not gif_file.lower().endswith(".gif"):
            continue
        
        num_frames, sizes = get_gif_info(gif_file, input_dir=input_dir)
        width, height = sizes[0] # assumes all frames have the same size

        for rank in rank_list:
            for thread in thread_list:
                for node in node_list:
                    for cuda in cuda_list:
                        times = {}
                        # MPI
                        try:
                            times["MPI"]  = profile_mpi(gif_file, gif_file, rank, node)
                        except:
                            times["MPI"] = np.inf
                        # OMP
                        try:
                            times["OMP"]  = profile_omp(gif_file, gif_file, thread)
                        except:
                            times["OMP"] = np.inf
                        # HYBRID
                        try:
                            times["HYB"]  = profile_hyb(gif_file, gif_file, rank, thread, node)
                        except:
                            times["HYB"] = np.inf
                        # CUDA
                        try:
                            if cuda:
                                times["CUDA"] = profile_cuda(gif_file, gif_file)
                            else:
                                times["CUDA"] = np.inf
                        except:
                            times["CUDA"] = np.inf

                        best_impl = min(times, key=times.get)
                        labels = {"MPI":0, "OMP":1, "HYB":2, "CUDA":3}
                        best_label = labels[best_impl]

                        features = [num_frames, width, height, rank, thread, node, cuda]

                        X.append(features)
                        y.append(best_label)

    return np.array(X), np.array(y)

def build_tree(X, y, max_depth):
    tree = DecisionTreeClassifier(max_depth=max_depth)
    tree.fit(X,y)
    return tree

def print_tree(tree):
    tree_text = export_text(tree, feature_names=feature_names)
    print("Decision tree :")
    print(tree_text)
    

ranks = [1,2,4]
threads = [2,4]
nodes = [1,2]
cudas = [True, False]

max_depth = 5 # TODO : test

X, y = create_data_set(INPUT_DIRECTORY, ranks, threads, nodes, cudas)

tree = build_tree(X, y, 5)

print_tree(tree)


100%|██████████| 18/18 [22:54<00:00, 76.39s/it]

Decision tree :
|--- num_frames <= 14.50
|   |--- width <= 1702.00
|   |   |--- width <= 542.50
|   |   |   |--- width <= 400.00
|   |   |   |   |--- width <= 260.00
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- width >  260.00
|   |   |   |   |   |--- class: 1
|   |   |   |--- width >  400.00
|   |   |   |   |--- ranks <= 1.50
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- ranks >  1.50
|   |   |   |   |   |--- class: 1
|   |   |--- width >  542.50
|   |   |   |--- threads <= 3.00
|   |   |   |   |--- height <= 756.50
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- height >  756.50
|   |   |   |   |   |--- class: 1
|   |   |   |--- threads >  3.00
|   |   |   |   |--- height <= 525.00
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- height >  525.00
|   |   |   |   |   |--- class: 1
|   |--- width >  1702.00
|   |   |--- class: 1
|--- num_frames >  14.50
|   |--- class: 1



In [7]:
print(y)

[1 1 1 1 2 2 1 1 1 2 1 1 1 1 2 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 2 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 2 2 1 1 2 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 1 1 1 1 1 2 2
 1 2 1 1 2 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 2 2 1 2
 1 1 1 1 1 2 1 1 1 1 1 1 2 2 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 2 2 1 1 1 1 1 2 1 2 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
